In [0]:
import pandas as pd

In [0]:
df=spark.read.table("post_renewal_churn.cleaned_dataset.features_table").toPandas()

In [0]:
df.columns

In [0]:
df.isnull().sum()

# Convert Target

In [0]:
df['label'] = df['prospect_outcome']

# Drop unnecessary columns

In [0]:
df_model = df.drop(['prospect_outcome'], axis=1)

# Handle Categorical Variables

In [0]:
df_model = pd.get_dummies(
    df_model,
    columns=[
        'Call_Direction',
        'Renewal_Impact_Due_to_Price_Increase',
        'Band',
        'score_category'
    ],
    drop_first=True
)

# Split X and y

In [0]:
X = df_model.drop('label', axis=1)
y = df_model['label']

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# RANDOM FOREST

In [0]:
%pip install xgboost

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

USE BOOSTING (BETTER PERFORMANCE)

In [0]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier()

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

print("GB Accuracy:", accuracy_score(y_test, y_pred_gb))
print("\nGB Report:\n", classification_report(y_test, y_pred_gb))
print("GB ROC-AUC:", roc_auc_score(y_test, y_prob_gb))

# XGBoost

In [0]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb = XGBClassifier(
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nXGB Report:\n", classification_report(y_test, y_pred_xgb))
print("XGB ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

# Metrices

## ROC Curve

In [0]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Probabilities
y_prob = xgb.predict_proba(X_test)[:, 1]

# ROC values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Plot
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0,1], [0,1], linestyle='--')  # random line

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Confusion Matrix

In [0]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Precision-Recall Curve

In [0]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure()
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

## Feature Importance

In [0]:
import pandas as pd

importance = pd.Series(xgb.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10).plot(kind='barh')

plt.title("Top Features")
plt.show()

## Threshold vs Recall

In [0]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.1)
recalls = []

for t in thresholds:
    y_pred_t = (y_prob > t).astype(int)
    from sklearn.metrics import recall_score
    recalls.append(recall_score(y_test, y_pred_t))

plt.plot(thresholds, recalls)
plt.xlabel("Threshold")
plt.ylabel("Recall")
plt.title("Threshold vs Recall")
plt.show()

In [0]:
%pip install shap

# SHAP

In [0]:
import shap

explainer = shap.Explainer(xgb)
shap_values = explainer(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test)

In [0]:
X_full_scaled = scaler.transform(X)

In [0]:
df_model_copy = df_model.copy()

df_model_copy['churn_probability'] = xgb.predict_proba(X_full_scaled)[:, 1]
df_model_copy['prediction'] = (df_model_copy['churn_probability'] > 0.5).astype(int)

In [0]:
df_final_dashboard = df.copy()

df_final_dashboard['churn_probability'] = df_model_copy['churn_probability']
df_final_dashboard['prediction'] = df_model_copy['prediction']

In [0]:
df_final_dashboard.to_csv("churn_dashboard.csv", index=False)